In [1]:
from google.colab import files

uploaded = files.upload()

Saving adult.zip to adult.zip


In [2]:
import zipfile
import os

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall("adult_dataset")

print("Dataset extracted successfully!")
print(os.listdir("adult_dataset"))

Dataset extracted successfully!
['adult.names', 'Index', 'adult.data', 'adult.test', 'old.adult.names']


In [3]:
import pandas as pd
import numpy as np

In [4]:
columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

df = pd.read_csv(
    "adult_dataset/adult.data",
    names=columns,
    skipinitialspace=True
)

print("Dataset loaded successfully!")
print("Original shape:", df.shape)

df.head()

Dataset loaded successfully!
Original shape: (32561, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [5]:
print("===== DATASET INFORMATION =====")
df.info()

print("\n===== DATASET SHAPE =====")
print(df.shape)

print("\n===== DUPLICATES =====")
print(df.duplicated().sum())

print("\n===== MISSING VALUES =====")
print(df.isnull().sum())

===== DATASET INFORMATION =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB

===== DATASET SHAPE =====
(32561, 15)

===== DUPLICATES ====

In [6]:
print("Missing values represented by '?':")

for column in df.columns:
    count = (df[column] == "?").sum()
    if count > 0:
        print(column, ":", count)

Missing values represented by '?':
workclass : 1836
occupation : 1843
native_country : 583


In [7]:
# Replace '?' with proper missing values
df = df.replace("?", pd.NA)

# Fill missing categorical values using mode
categorical_columns = [
    "workclass",
    "occupation",
    "native_country"
]

for column in categorical_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64


In [8]:
duplicates_before = df.duplicated().sum()

df = df.drop_duplicates()

duplicates_after = df.duplicated().sum()

print("Duplicates before removal:", duplicates_before)
print("Duplicates after removal:", duplicates_after)
print("Shape after duplicate removal:", df.shape)

Duplicates before removal: 24
Duplicates after removal: 0
Shape after duplicate removal: (32537, 15)


In [9]:
print("Data types:")
print(df.dtypes)

Data types:
age                int64
workclass         object
fnlwgt             int64
education         object
education_num      int64
marital_status    object
occupation        object
relationship      object
race              object
sex               object
capital_gain       int64
capital_loss       int64
hours_per_week     int64
native_country    object
income            object
dtype: object


In [10]:
numerical_columns = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

print("===== OUTLIER DETECTION USING IQR =====")

for column in numerical_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print(column, ":", len(outliers), "outliers")

===== OUTLIER DETECTION USING IQR =====
age : 142 outliers
fnlwgt : 993 outliers
education_num : 1193 outliers
capital_gain : 2712 outliers
capital_loss : 1519 outliers
hours_per_week : 9002 outliers


In [12]:
# Apply IQR capping only to selected numerical features

outlier_columns = [
    "fnlwgt",
    "hours_per_week"
]

for column in outlier_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[column] = df[column].clip(
        lower=lower_bound,
        upper=upper_bound
    )

print("Selected outliers handled successfully.")

Selected outliers handled successfully.


In [13]:
# Feature 1: Age group

df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 50, 65, 100],
    labels=[
        "Young",
        "Young Adult",
        "Adult",
        "Middle Age",
        "Senior"
    ]
)

# Feature 2: Work hours category

df["work_hours_category"] = pd.cut(
    df["hours_per_week"],
    bins=[0, 35, 40, 60, 100],
    labels=[
        "Part-time",
        "Standard",
        "Overtime",
        "Very High"
    ]
)

# Feature 3: Income encoding

df["income_encoded"] = df["income"].map({
    "<=50K": 0,
    ">50K": 1
})

print("Feature engineering completed!")

df[
    [
        "age",
        "age_group",
        "hours_per_week",
        "work_hours_category",
        "income",
        "income_encoded"
    ]
].head(10)

Feature engineering completed!


,age,age_group,hours_per_week,work_hours_category,income,income_encoded
0,39,Adult,40.0,Standard,<=50K,0
1,50,Adult,32.5,Part-time,<=50K,0
2,38,Adult,40.0,Standard,<=50K,0
3,53,Middle Age,40.0,Standard,<=50K,0
4,28,Young Adult,40.0,Standard,<=50K,0
5,37,Adult,40.0,Standard,<=50K,0
6,49,Adult,32.5,Part-time,<=50K,0
7,52,Middle Age,45.0,Overtime,>50K,1
8,31,Young Adult,50.0,Overtime,>50K,1
9,42,Adult,40.0,Standard,>50K,1


In [14]:
print("========== FINAL DATASET ==========")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

========== FINAL DATASET ==========
Rows: 32537
Columns: 18

Missing values:
age                    0
workclass              0
fnlwgt                 0
education              0
education_num          0
marital_status         0
occupation             0
relationship           0
race                   0
sex                    0
capital_gain           0
capital_loss           0
hours_per_week         0
native_country         0
income                 0
age_group              0
work_hours_category    0
income_encoded         0
dtype: int64

Duplicate rows:
24

Data types:
age                       int64
workclass                object
fnlwgt                    int64
education                object
education_num             int64
marital_status           object
occupation               object
relationship             object
race                     object
sex                      object
capital_gain              int64
capital_loss              int64
hours_per_week          float64
native_coun

In [15]:
clean_file = "adult_income_clean.csv"

df.to_csv(clean_file, index=False)

print("Clean dataset saved successfully!")
print("File name:", clean_file)

Clean dataset saved successfully!
File name: adult_income_clean.csv


In [16]:
from google.colab import files

files.download("adult_income_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
print("""
========================================
DATA PREPROCESSING COMPLETED
========================================

Dataset: UCI Adult Income Dataset

Original records: 32,561
Records after duplicate removal: 32,537

Original features: 15
Final features: 18

Missing values:
- '?' converted to missing values
- Missing categorical values filled using mode

Duplicates:
- Duplicate rows removed

Outliers:
- IQR method used for detection
- Selected numerical features capped

Feature Engineering:
- age_group
- work_hours_category
- income_encoded

Output:
adult_income_clean.csv
========================================
""")


DATA PREPROCESSING COMPLETED

Dataset: UCI Adult Income Dataset

Original records: 32,561
Records after duplicate removal: 32,537

Original features: 15
Final features: 18

Missing values:
- '?' converted to missing values
- Missing categorical values filled using mode

Duplicates:
- Duplicate rows removed

Outliers:
- IQR method used for detection
- Selected numerical features capped

Feature Engineering:
- age_group
- work_hours_category
- income_encoded

Output:
adult_income_clean.csv

